# Mangrove Pulse: review checks
Reproduce commit `a5131ce`, then measure the eligible-population edge share and reverse the calibration/evaluation halves. The reverse split is exploratory: both halves have already been inspected. These checks measure reference agreement and computational reproducibility, not field accuracy, drone performance or ecological benefit.
The notebook uses the repository's cached inputs; it does not request new satellite imagery.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import subprocess, sys, tempfile, time, json, hashlib, os, zipfile
os.environ['MPLBACKEND'] = 'Agg'
work = Path(tempfile.mkdtemp(prefix='mangrove_review_'))
repo, out = work / 'repository', work / 'reproduction'
evidence = work / 'evidence'; evidence.mkdir()
expected_commit = 'a5131ce0b702b6018274c7d20af1346eee43654d'
record = {'status': 'running', 'commit': expected_commit, 'python': sys.version, 'steps': {}}
def step(name, command):
    start = time.perf_counter()
    with (evidence / (name + '.log')).open('w') as log:
        completed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, timeout=600, env=os.environ.copy())
    record['steps'][name] = {'seconds': time.perf_counter()-start, 'returncode': completed.returncode}
    print(name, record['steps'][name], flush=True)
    if completed.returncode: raise RuntimeError((evidence / (name + '.log')).read_text()[-5000:])
try:
    step('clone', ['git', 'clone', '--depth', '1', 'https://github.com/Salim-Ammar/mangrove-pulse.git', str(repo)])
    step('fetch_candidate', ['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', expected_commit])
    step('checkout_candidate', ['git', '-C', str(repo), 'checkout', '--detach', expected_commit])
    actual = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != expected_commit: raise ValueError('Repository HEAD changed; identify the tested candidate before proceeding: '+actual)
    step('install', [sys.executable, '-m', 'pip', 'install', '-r', str(repo / 'requirements.txt')])
    import importlib.metadata
    record['packages'] = {name: importlib.metadata.version(name) for name in ['numpy', 'rasterio', 'matplotlib']}
    step('standalone', [sys.executable, str(repo / 'src/run_submission.py'), '--root', str(repo), '--out', str(out)])
    step('sensitivity', [sys.executable, str(repo / 'src/review_sensitivity.py'), '--baseline', str(out / 'baseline'), '--out', str(work / 'sensitivity')])
    report = json.loads((work / 'sensitivity/review_sensitivity.json').read_text())
    print(json.dumps({k:v for k,v in report.items() if k != 'input_sha256'}, indent=2))
    record['status'] = 'succeeded'
except Exception as exc:
    record.update(status='failed', error=str(exc)); print('FAILED:', exc)
finally:
    record['repository_manifest_sha256'] = hashlib.sha256((repo / 'SHA256SUMS.json').read_bytes()).hexdigest() if (repo / 'SHA256SUMS.json').exists() else None
    (evidence / 'execution_record.json').write_text(json.dumps(record, indent=2)+'\n')


## Preserve the evidence
The download includes logs, runtime versions, reproduced summaries and the sensitivity report. If a step failed, it is a diagnostic package, not a successful result.

In [ ]:
package = work / 'Mangrove_Pulse_Review_Checks_Results.zip'
with zipfile.ZipFile(package, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in [evidence, out, work / 'sensitivity']:
        if folder.exists():
            for path in sorted(folder.rglob('*')):
                if path.is_file() and 'inputs' not in path.relative_to(folder).parts and path.suffix in ['.json', '.log', '.txt', '.csv']:
                    archive.write(path, path.relative_to(work))
print('Status:', record['status'], 'bytes:', package.stat().st_size, 'SHA-256:', hashlib.sha256(package.read_bytes()).hexdigest())
files.download(str(package))
